# E1: Typed Failure Telemetry vs 1-bit Feedback (miniF2F, Colab)

Runs the pre-registered E1 experiment (`docs/experiments/e1_preregistration.md`):
three feedback arms (`a0_onebit`, `a1_raw_error`, `a2_typed_packet`) under an
identical model, instruction text, and per-theorem call budget, audited against Lean.

Decision rule (fixed before execution): the paired bootstrap 95% CI of
`solve_rate(a2) - solve_rate(a1)` must exclude zero in favor of a2.

Order of cells: setup -> dry smoke (no API cost) -> Lean toolchain -> real run -> report.

In [ ]:
# 1. Clone the repo and install
REPO_URL = "https://github.com/abhorrence-of-Gods/lean-rgc-automation-stack.git"
import os
if not os.path.exists("lean-rgc-automation-stack"):
    !git clone --depth 1 {REPO_URL}
%cd lean-rgc-automation-stack
!pip install -q -e .
!lean-rgc --help | head -5

In [ ]:
# 2. Optional: mount Drive so the audit cache and LLM cache survive session resets
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/lean_rgc_e1"
else:
    PERSIST = "/content/lean_rgc_e1"
WORK_ROOT = "/content/lean_rgc_e1_work"
MINIF2F_REPO = f"{WORK_ROOT}/minif2f"
os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(f"{PERSIST}/llm_cache", exist_ok=True)
os.makedirs(f"{PERSIST}/runs", exist_ok=True)
AUDIT_CACHE_DB = f"{PERSIST}/audit_cache.sqlite"
print("persist dir:", PERSIST)
print("work dir:", WORK_ROOT)

In [ ]:
# 3. API key from Colab Secrets (left panel -> key icon -> add LEAN_RGC_LLM_API_KEY)
from google.colab import userdata
os.environ["LEAN_RGC_LLM_API_KEY"] = userdata.get("LEAN_RGC_LLM_API_KEY")
print("api key loaded:", bool(os.environ.get("LEAN_RGC_LLM_API_KEY")))

In [ ]:
# 4. LLM config (GitHub Models via OpenAI-compatible chat completions)
import json
LLM_CONFIG = {
    "provider": "openai_compatible",
    "model": "openai/gpt-4.1",
    "base_url": "https://models.github.ai/inference",
    "api_key_env": "LEAN_RGC_LLM_API_KEY",
    "temperature": 0.2,
    "top_p": 0.95,
    "max_tokens": 1024,
    "seed": 0,
    "cache_dir": f"{PERSIST}/llm_cache",
}
with open("llm.json", "w") as f:
    json.dump(LLM_CONFIG, f, indent=2)
MOCK_CONFIG = {**LLM_CONFIG, "provider": "mock", "mock_responses": [
    json.dumps({"proposals": [{"proposal_kind": "tactic", "lean_tactic": "simp"}]})
]}
with open("llm_mock.json", "w") as f:
    json.dump(MOCK_CONFIG, f, indent=2)
print(json.dumps(LLM_CONFIG, indent=2))

In [ ]:
# 5. Plumbing smoke (zero API cost, no Lean): mock LLM + dry-run audit on 2 tiny tasks
smoke_tasks = [
    {"task_id": "smoke0", "statement": "True", "imports": []},
    {"task_id": "smoke1", "statement": "1 = 1", "imports": []},
]
with open("smoke_tasks.jsonl", "w") as f:
    f.write("".join(json.dumps(t) + "\n" for t in smoke_tasks))
!lean-rgc eval-run --tasks smoke_tasks.jsonl --arm a1_raw_error \
  --llm-config llm_mock.json --out-dir {PERSIST}/runs/smoke_a1 --run-id smoke_a1 \
  --budget-calls 2 --dry-run --lean-cmd "echo lean" | tail -20

In [ ]:
# 6. Lean toolchain + miniF2F (long: toolchain ~5min, mathlib cache download ~10-20min)
!curl -sSfL https://github.com/leanprover/elan/releases/latest/download/elan-x86_64-unknown-linux-gnu.tar.gz | tar xz
!./elan-init -y --default-toolchain none
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
!lean-rgc minif2f-fetch --out {MINIF2F_REPO} --summary-out {PERSIST}/minif2f_fetch.json
%cd {MINIF2F_REPO}
!~/.elan/bin/lake exe cache get
!~/.elan/bin/lake build
%cd /content/lean-rgc-automation-stack

In [ ]:
# 7. Build the pre-registered task subset (100 miniF2F-test theorems, seed 0)
!lean-rgc minif2f-tasks --repo {MINIF2F_REPO} --split test \
  --out {PERSIST}/minif2f_test_tasks.jsonl --summary-out {PERSIST}/minif2f_tasks_summary.json
!lean-rgc eval-subset --tasks {PERSIST}/minif2f_test_tasks.jsonl \
  --out {PERSIST}/e1_tasks.jsonl --n 100 --seed 0

In [ ]:
# 8. E1: run the three arms (identical budget; audit cache shared across arms per lane rules)
LEAN_CMD = "~/.elan/bin/lake env lean"
WORKDIR = MINIF2F_REPO
for arm in ["a0_onebit", "a1_raw_error", "a2_typed_packet"]:
    print("=== arm:", arm, "===")
    !lean-rgc eval-run --tasks {PERSIST}/e1_tasks.jsonl --arm {arm} \
      --llm-config llm.json --out-dir {PERSIST}/runs/e1_{arm} --run-id e1_{arm} \
      --budget-calls 8 --max-proposals 4 \
      --lean-cmd "{LEAN_CMD}" --workdir {WORKDIR} --import-mode preserve \
      --queue-backend bulk --workers 4 --job-timeout-s 180 \
      --audit-cache-db {AUDIT_CACHE_DB} | tail -15

In [ ]:
# 9. Pre-registered report: paired bootstrap over the identical task set
!lean-rgc eval-report \
  --episodes a0_onebit={PERSIST}/runs/e1_a0_onebit/episodes.jsonl \
             a1_raw_error={PERSIST}/runs/e1_a1_raw_error/episodes.jsonl \
             a2_typed_packet={PERSIST}/runs/e1_a2_typed_packet/episodes.jsonl \
  --out {PERSIST}/runs/e1_report.json --n-bootstrap 10000 --seed 0
report = json.load(open(f"{PERSIST}/runs/e1_report.json"))
for comp in report["paired_comparisons"]:
    verdict = "CI EXCLUDES ZERO" if comp["ci_excludes_zero"] else "inconclusive"
    print(f"{comp['arm_a']} - {comp['arm_b']}: delta={comp['mean_delta']:+.3f} "
          f"[{comp['ci_low']:+.3f}, {comp['ci_high']:+.3f}] {verdict}")

In [ ]:
# 10. Episode store summary (boundary cache rate, per-arm solve rates)
from lean_rgc.pbct.episode_store import summarize_prompt_db
for arm in ["a0_onebit", "a1_raw_error", "a2_typed_packet"]:
    db = f"{PERSIST}/runs/e1_{arm}/prompt.sqlite"
    if os.path.exists(db):
        print(arm, json.dumps(summarize_prompt_db(db)["episodes_by_arm"], indent=2))